# Análise de dados TCP-CII

### Importação dos parâmetros universais

In [76]:
import importlib.util
from pathlib import Path

path = Path("../../../parametros/config.py").resolve()

spec = importlib.util.spec_from_file_location("parametros", path)
parametros = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parametros)

In [77]:
# Parâmetros importados do arquivo config.py
print(
    "Filtrar por quantidade de alelos TCC2:.........................",
    parametros.filtarar_por_qte_de_alelos_tcc2,
)
print(
    "Parâmetro de filtragem median binding percentile TCC2:.........",
    parametros.parametro_de_filtragem_mbp_tcc2,
)
print(
    "Percentual de match mínimo TCC2:...............................",
    parametros.percent_match_minimo_tcc2,
)

Filtrar por quantidade de alelos TCC2:......................... 2
Parâmetro de filtragem median binding percentile TCC2:......... 1
Percentual de match mínimo TCC2:............................... 95.0


In [78]:
import pandas as pd

In [79]:
df = pd.read_csv('./T CELL/DENV 2 - T Cell Prediction - Class II.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,LNDTWKIEKASFIEVK,206,221,16,HLA-DRB1*01:01,246,0.02,WKIEKASFI,0.982841,0.02
1,1,LNDTWKIEKASFIEVKN,206,222,17,HLA-DRB1*01:01,314,0.06,WKIEKASFI,0.971760,0.06
2,1,LNDTWKIEKASFIEV,206,220,15,HLA-DRB1*01:01,178,0.08,WKIEKASFI,0.970218,0.08
3,1,DSGCVVSWKNKELK,1,14,14,HLA-DRB1*15:01,69,0.11,VVSWKNKEL,0.892845,0.11
4,1,LNDTWKIEKASFIEVKNC,206,223,18,HLA-DRB1*01:01,382,0.12,WKIEKASFI,0.942143,0.12
...,...,...,...,...,...,...,...,...,...,...,...
16411,1,TTASGKLITEWCCRSCTLP,301,319,19,HLA-DRB3*02:02,468,100.00,LITEWCCRS,0.000010,100.00
16412,1,TTASGKLITEWCCRSCTLPP,301,320,20,HLA-DRB3*02:02,535,100.00,LITEWCCRS,0.000007,100.00
16413,1,KLITEWCCRSCTL,306,318,13,HLA-DRB3*02:02,62,100.00,LITEWCCRS,0.000007,100.00
16414,1,TTASGKLITEWCC,301,313,13,HLA-DRB3*02:02,61,100.00,TASGKLITE,0.000006,100.00


## Selecionando Epítopos por median binding percentile.

In [80]:
mbp_minimo = parametros.parametro_de_filtragem_mbp_tcc2

In [81]:
df_mbp_m5 = df[df['median binding percentile'] < mbp_minimo].copy()
print("Filtrando por median binding percentile < ", mbp_minimo)
df_mbp_m5

Filtrando por median binding percentile <  1


,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,LNDTWKIEKASFIEVK,206,221,16,HLA-DRB1*01:01,246,0.02,WKIEKASFI,0.982841,0.02
1,1,LNDTWKIEKASFIEVKN,206,222,17,HLA-DRB1*01:01,314,0.06,WKIEKASFI,0.971760,0.06
2,1,LNDTWKIEKASFIEV,206,220,15,HLA-DRB1*01:01,178,0.08,WKIEKASFI,0.970218,0.08
3,1,DSGCVVSWKNKELK,1,14,14,HLA-DRB1*15:01,69,0.11,VVSWKNKEL,0.892845,0.11
4,1,LNDTWKIEKASFIEVKNC,206,223,18,HLA-DRB1*01:01,382,0.12,WKIEKASFI,0.942143,0.12
...,...,...,...,...,...,...,...,...,...,...,...
101,1,QYKFQPESPSKLASAIQ,31,47,17,HLA-DRB1*01:01,279,0.94,FQPESPSKL,0.796321,0.94
102,1,GSGIFITDNVHTWTEQYK,16,33,18,HLA-DRB3*01:01,344,0.94,FITDNVHTW,0.403507,0.94
103,1,LNDTWKIEKASFIEVKN,206,222,17,HLA-DPA1*01:03/DPB1*04:01,314,0.94,IEKASFIEV,0.276922,0.94
104,1,GVFTTNIWLKLKEKQ,161,175,15,HLA-DPA1*03:01/DPB1*04:02,169,0.96,FTTNIWLKL,0.292246,0.96


## Agrupando por pepitideos e agregando colunas pertinentes

In [82]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    )
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDNRAVHADM,186,198,1,0.12,HLA-DRB3*02:02
1,AAIKDNRAVHADMG,186,199,1,0.16,HLA-DRB3*02:02
2,AAIKDNRAVHADMGY,186,200,1,0.32,HLA-DRB3*02:02
3,AAIKDNRAVHADMGYW,186,201,1,0.53,HLA-DRB3*02:02
4,AAIKDNRAVHADMGYWI,186,202,1,0.79,HLA-DRB3*02:02
...,...,...,...,...,...,...
77,VSQHNYRPGYHTQITGPWHLG,251,271,1,0.76,HLA-DQA1*05:01/DQB1*02:01
78,WIESALNDTWKIEKASFIEV,201,220,1,0.81,HLA-DRB1*01:01
79,WIESALNDTWKIEKASFIEVK,201,221,1,0.12,HLA-DRB1*01:01
80,YRPGYHTQITGPW,256,268,1,0.77,HLA-DQA1*03:01/DQB1*03:02


## Filtragem por qte_de_alelos

In [83]:
# mbp_minimo = 1
qte_alelos_minimo = parametros.filtarar_por_qte_de_alelos_tcc2

In [84]:
# Filtro do número de alelos
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= qte_alelos_minimo
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AECPNTNRAWNSLEVEDYGF,141,160,2,0.610,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*05:01/DQB1..."
1,AECPNTNRAWNSLEVEDYGFG,141,161,2,0.335,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*05:01/DQB1..."
2,EDYGFGVFTTNIWLKLKEK,156,174,3,0.770,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
3,EDYGFGVFTTNIWLKLKEKQ,156,175,2,0.830,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
4,EDYGFGVFTTNIWLKLKEKQD,156,176,2,0.885,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
5,GSGIFITDNVHTWT,16,29,2,0.695,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01"
6,GSGIFITDNVHTWTE,16,30,2,0.445,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01"
7,GSGIFITDNVHTWTEQ,16,31,2,0.330,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01"
8,GSGIFITDNVHTWTEQY,16,32,2,0.480,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01"
9,GSGIFITDNVHTWTEQYK,16,33,2,0.680,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01"


## Sorting por median_biding_percentile

In [85]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,GSGIFITDNVHTWTEQ,16,31,2,0.330,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01"
1,GVFTTNIWLKLKE,161,173,4,0.335,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
2,AECPNTNRAWNSLEVEDYGFG,141,161,2,0.335,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*05:01/DQB1..."
3,GVFTTNIWLKLKEK,161,174,4,0.350,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
4,LNDTWKIEKASFIEVK,206,221,2,0.440,"HLA-DRB1*01:01, HLA-DRB1*07:01"
5,GSGIFITDNVHTWTE,16,30,2,0.445,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01"
6,GSGIFITDNVHTWTEQY,16,32,2,0.480,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01"
7,LNDTWKIEKASFIEVKN,206,222,2,0.500,"HLA-DPA1*01:03/DPB1*04:01, HLA-DRB1*01:01"
8,TNRAWNSLEVEDYG,146,159,2,0.560,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
9,AECPNTNRAWNSLEVEDYGF,141,160,2,0.610,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*05:01/DQB1..."


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [86]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos_tcell_2.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0          GSGIFITDNVHTWTEQ
1             GVFTTNIWLKLKE
2     AECPNTNRAWNSLEVEDYGFG
3            GVFTTNIWLKLKEK
4          LNDTWKIEKASFIEVK
5           GSGIFITDNVHTWTE
6         GSGIFITDNVHTWTEQY
7         LNDTWKIEKASFIEVKN
8            TNRAWNSLEVEDYG
9      AECPNTNRAWNSLEVEDYGF
10       GSGIFITDNVHTWTEQYK
11           GSGIFITDNVHTWT
12            KRSLRPQPTELKY
13      EDYGFGVFTTNIWLKLKEK
14           YRPGYHTQITGPWH
15           KRSLRPQPTELKYS
16     EDYGFGVFTTNIWLKLKEKQ
17          GVFTTNIWLKLKEKQ
18    EDYGFGVFTTNIWLKLKEKQD
Name: peptide, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [87]:
# !seqkit grep -s -v -r -p '[-*]' './Fastas/denv1_NS1_proteinas.fa' > DENV1_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [88]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_2.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,GSGIFITDNVHTWTEQ,16,97.00% (840/866),93.75%,100.00%,NaN
1,2,NP 2,GVFTTNIWLKLKE,13,58.20% (504/866),84.62%,100.00%,NaN
2,3,NP 3,AECPNTNRAWNSLEVEDYGFG,21,93.19% (807/866),90.48%,100.00%,NaN
3,4,NP 4,GVFTTNIWLKLKEK,14,23.09% (200/866),78.57%,100.00%,NaN
4,5,NP 5,LNDTWKIEKASFIEVK,16,54.16% (469/866),56.25%,100.00%,NaN
5,6,NP 6,GSGIFITDNVHTWTE,15,97.00% (840/866),93.33%,100.00%,NaN
6,7,NP 7,GSGIFITDNVHTWTEQY,17,97.00% (840/866),94.12%,100.00%,NaN
7,8,NP 8,LNDTWKIEKASFIEVKN,17,21.48% (186/866),52.94%,100.00%,NaN
8,9,NP 9,TNRAWNSLEVEDYG,14,96.19% (833/866),92.86%,100.00%,NaN
9,10,NP 10,AECPNTNRAWNSLEVEDYGF,20,93.19% (807/866),90.00%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [89]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# MPB
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "median_binding_percentile"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# start
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "start"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# end
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "end"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos
0,NP 1,GSGIFITDNVHTWTEQ,16,97.00% (840/866),93.75%,100.00%,2,0.330,16,31,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01"
1,NP 2,GVFTTNIWLKLKE,13,58.20% (504/866),84.62%,100.00%,4,0.335,161,173,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
2,NP 3,AECPNTNRAWNSLEVEDYGFG,21,93.19% (807/866),90.48%,100.00%,2,0.335,141,161,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*05:01/DQB1..."
3,NP 4,GVFTTNIWLKLKEK,14,23.09% (200/866),78.57%,100.00%,4,0.350,161,174,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
4,NP 5,LNDTWKIEKASFIEVK,16,54.16% (469/866),56.25%,100.00%,2,0.440,206,221,"HLA-DRB1*01:01, HLA-DRB1*07:01"
5,NP 6,GSGIFITDNVHTWTE,15,97.00% (840/866),93.33%,100.00%,2,0.445,16,30,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01"
6,NP 7,GSGIFITDNVHTWTEQY,17,97.00% (840/866),94.12%,100.00%,2,0.480,16,32,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01"
7,NP 8,LNDTWKIEKASFIEVKN,17,21.48% (186/866),52.94%,100.00%,2,0.500,206,222,"HLA-DPA1*01:03/DPB1*04:01, HLA-DRB1*01:01"
8,NP 9,TNRAWNSLEVEDYG,14,96.19% (833/866),92.86%,100.00%,2,0.560,146,159,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
9,NP 10,AECPNTNRAWNSLEVEDYGF,20,93.19% (807/866),90.00%,100.00%,2,0.610,141,160,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*05:01/DQB1..."


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [90]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos,percent_match
0,NP 1,GSGIFITDNVHTWTEQ,16,97.00% (840/866),93.75%,100.00%,2,0.330,16,31,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",97.00
1,NP 2,GVFTTNIWLKLKE,13,58.20% (504/866),84.62%,100.00%,4,0.335,161,173,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1...",58.20
2,NP 3,AECPNTNRAWNSLEVEDYGFG,21,93.19% (807/866),90.48%,100.00%,2,0.335,141,161,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*05:01/DQB1...",93.19
3,NP 4,GVFTTNIWLKLKEK,14,23.09% (200/866),78.57%,100.00%,4,0.350,161,174,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1...",23.09
4,NP 5,LNDTWKIEKASFIEVK,16,54.16% (469/866),56.25%,100.00%,2,0.440,206,221,"HLA-DRB1*01:01, HLA-DRB1*07:01",54.16
5,NP 6,GSGIFITDNVHTWTE,15,97.00% (840/866),93.33%,100.00%,2,0.445,16,30,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",97.00
6,NP 7,GSGIFITDNVHTWTEQY,17,97.00% (840/866),94.12%,100.00%,2,0.480,16,32,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",97.00
7,NP 8,LNDTWKIEKASFIEVKN,17,21.48% (186/866),52.94%,100.00%,2,0.500,206,222,"HLA-DPA1*01:03/DPB1*04:01, HLA-DRB1*01:01",21.48
8,NP 9,TNRAWNSLEVEDYG,14,96.19% (833/866),92.86%,100.00%,2,0.560,146,159,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1...",96.19
9,NP 10,AECPNTNRAWNSLEVEDYGF,20,93.19% (807/866),90.00%,100.00%,2,0.610,141,160,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*05:01/DQB1...",93.19


### Sort e filtragem por percent_match

In [91]:
minimum_percent_match = parametros.percent_match_minimo_tcc2

In [92]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= minimum_percent_match]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos,percent_match
0,NP 1,GSGIFITDNVHTWTEQ,16,97.00% (840/866),93.75%,100.00%,2,0.330,16,31,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",97.00
1,NP 6,GSGIFITDNVHTWTE,15,97.00% (840/866),93.33%,100.00%,2,0.445,16,30,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",97.00
2,NP 7,GSGIFITDNVHTWTEQY,17,97.00% (840/866),94.12%,100.00%,2,0.480,16,32,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",97.00
3,NP 12,GSGIFITDNVHTWT,14,97.00% (840/866),92.86%,100.00%,2,0.695,16,29,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",97.00
4,NP 11,GSGIFITDNVHTWTEQYK,18,96.77% (838/866),94.44%,100.00%,2,0.680,16,33,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",96.77
5,NP 9,TNRAWNSLEVEDYG,14,96.19% (833/866),92.86%,100.00%,2,0.560,146,159,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1...",96.19


### Estrutura secundária

In [93]:
def processar_horiz(caminho_horiz):
    """
    Lê o arquivo .horiz do PSIPRED e retorna um dicionário 
    contendo as strings de confiança ('conf'), predição ('pred') e sequência ('seq').
    """
    confianca = ""
    predicao = ""
    sequencia = ""
    
    with open(caminho_horiz, 'r') as f:
        for linha in f:
            linha_str = linha.strip()
            if linha_str.startswith("Conf:"):
                confianca += linha_str.split(":", 1)[1].strip()
            elif linha_str.startswith("Pred:"):
                predicao += linha_str.split(":", 1)[1].strip()
            elif linha_str.startswith("AA:"):
                sequencia += linha_str.split(":", 1)[1].strip()
                
    return {
        "conf": confianca,
        "pred": predicao,
        "seq": sequencia
    }

In [94]:
denv_horiz = processar_horiz("./PSIPRED/denv.horiz")
denv_horiz

{'conf': '9024886999949816069996771222301657509999999999999982987778899899999997799999998866962899959832555020133336998640134324513003268789379981899887667661785699998476787434999865333556992778778108988983374589873288079999999501863179210578888641004368566598553446899653468895238289988317999899668989997541135168962563421679999859996994060304650467854302403409',
 'pred': 'CCCCEEECCCCEEEECEEEEEECCCCCCCCCCEEECCCHHHHHHHHHHHHHHCCCCCCCCCHHHHHHHHHHHHHHHHHHHHCCCEEEEEECCCEEEEECCCCCCCCCCCCCCCCCCCCCCCCEECCCCCCEEEEEECCCCCCCCCCCEEEEEEEEEEEEEEEEECEEEEEECCCCCCCCCHHHHHHHHCCCEEEEECCCEEEEEECCCCEEEEEEEEEEEEEEECCCHHCCCCCCCCCCCCCCCHHHCCCCCCCCCCCCCCCCCCCCCCCCCEEEEEEECCCCEEEECCCCCCCCCCCEEEECCCCCCCCEECCCCCCCCEEEECCCCCCCCEEEEECCCCCCCEEEEEEEC',
 'seq': 'DSGCVVSWKNKELKCGSGIFITDNVHTWTEQYKFQPESPSKLASAIQKAHEEGICGIRSVTRLENLMWKQITPELNHILSENEVKLTIMTGDIKGIMQAGKRSLRPQPTELKYSWKTWGKAKMLSTESHNQTFLIDGPETAECPNTNRAWNSLEVEDYGFGVFTTNIWLKLKEKQDVFCDSKLMSAAIKDNRAVHADMGYWIESALNDTWKIEKASFIEVKNCHWPKSHTLWSNGVLESEMIIPKNLAGPVSQHNYRPGYH

In [95]:
def avaliar_epitopos_com_horiz(df_epitopos, conf, pred, seq):
    """Cruza o DataFrame de epítopos com as strings de predição do .horiz já processadas."""
    resultados = []
    
    for idx, row in df_epitopos.iterrows():
        # Identifica a coluna correta do peptídeo dependendo do estágio do pipeline
        col_pep = 'peptide' if 'peptide' in df_epitopos.columns else 'Epitope sequence'
        peptideo = row[col_pep]
        start = int(row['start'])
        end = int(row['end'])
        
        # Ajuste para índice em Python (base 0)
        idx_inicio = start - 1
        idx_fim = end
        
        # Extrai os trechos correspondentes nas strings do horiz
        sub_pred = pred[idx_inicio:idx_fim]
        sub_conf = conf[idx_inicio:idx_fim]
        
        # Conta a proporção de cada estrutura secundária
        # 'H' = Hélice, 'E' = Fita Beta, 'C' ou '.' = Coil/Alça
        n_coils = sub_pred.count('C') + sub_pred.count('.')
        n_helix = sub_pred.count('H')
        n_strand = sub_pred.count('E')
        
        resultados.append({
            'Epitope': peptideo,
            'Start': start,
            'End': end,
            'Pred_SS': sub_pred,
            'Coils': n_coils,
            'Helices': n_helix,
            'Strands': n_strand
        })
        
    return pd.DataFrame(resultados)

In [96]:
conf = denv_horiz["conf"]
pred = denv_horiz["pred"]
seq = denv_horiz["seq"]

avaliacao_epitopos = avaliar_epitopos_com_horiz(conservacy_result_filtered, conf, pred, seq)
avaliacao_epitopos

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands
0,GSGIFITDNVHTWTEQ,16,31,CEEEEEECCCCCCCCC,10,0,6
1,GSGIFITDNVHTWTE,16,30,CEEEEEECCCCCCCC,9,0,6
2,GSGIFITDNVHTWTEQY,16,32,CEEEEEECCCCCCCCCC,11,0,6
3,GSGIFITDNVHTWT,16,29,CEEEEEECCCCCCC,8,0,6
4,GSGIFITDNVHTWTEQYK,16,33,CEEEEEECCCCCCCCCCE,11,0,7
5,TNRAWNSLEVEDYG,146,159,CCEEEEEEEEEEEE,2,0,12


In [97]:
# Calcula o percentual de Coils (flexibilidade) para cada epítopo
avaliacao_epitopos['Percent_Coils'] = (
    avaliacao_epitopos['Coils'] / (avaliacao_epitopos['Coils'] + avaliacao_epitopos['Helices'] + avaliacao_epitopos['Strands'])
) * 100

# Ordena os epítopos do mais flexível (maior percentual de coils) para o mais rígido
epitopos_flexiveis = avaliacao_epitopos.sort_values(
    by=['Percent_Coils', 'Coils'], 
    ascending=[False, False]
).reset_index(drop=True)

epitopos_flexiveis

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils
0,GSGIFITDNVHTWTEQY,16,32,CEEEEEECCCCCCCCCC,11,0,6,64.705882
1,GSGIFITDNVHTWTEQ,16,31,CEEEEEECCCCCCCCC,10,0,6,62.500000
2,GSGIFITDNVHTWTEQYK,16,33,CEEEEEECCCCCCCCCCE,11,0,7,61.111111
3,GSGIFITDNVHTWTE,16,30,CEEEEEECCCCCCCC,9,0,6,60.000000
4,GSGIFITDNVHTWT,16,29,CEEEEEECCCCCCC,8,0,6,57.142857
5,TNRAWNSLEVEDYG,146,159,CCEEEEEEEEEEEE,2,0,12,14.285714


#### Filtrar epítopos que tenham pela % de sua estrutura composta por Coils

In [98]:
valor_corte_flexibilidade = parametros.corte_flexibilidade
print(f"Valor de corte de flexibilidade (Percent_Coils): {valor_corte_flexibilidade}%")

Valor de corte de flexibilidade (Percent_Coils): 35.0%


In [99]:
corte_flexibilidade = valor_corte_flexibilidade
epitopos_filtrados_flex = epitopos_flexiveis[epitopos_flexiveis['Percent_Coils'] >= corte_flexibilidade]

epitopos_filtrados_flex

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils
0,GSGIFITDNVHTWTEQY,16,32,CEEEEEECCCCCCCCCC,11,0,6,64.705882
1,GSGIFITDNVHTWTEQ,16,31,CEEEEEECCCCCCCCC,10,0,6,62.500000
2,GSGIFITDNVHTWTEQYK,16,33,CEEEEEECCCCCCCCCCE,11,0,7,61.111111
3,GSGIFITDNVHTWTE,16,30,CEEEEEECCCCCCCC,9,0,6,60.000000
4,GSGIFITDNVHTWT,16,29,CEEEEEECCCCCCC,8,0,6,57.142857


### Merge de dataframes

In [100]:
# Seleciona apenas as colunas exclusivas de conservação (excluindo 'start' e 'end' repetidos, se houver)
colunas_para_pegar = [c for c in conservacy_result_filtered.columns if c not in ['start', 'end', 'Epitope sequence']]
colunas_para_pegar.append('Epitope sequence') # Garante que a chave está incluída

# Realiza o merge limpo
df_consolidado = pd.merge(
    epitopos_filtrados_flex,
    conservacy_result_filtered[colunas_para_pegar],
    left_on='Epitope',
    right_on='Epitope sequence',
    how='inner'
)

# Remove se sobrou a coluna 'Epitope sequence' duplicada:
if 'Epitope sequence' in df_consolidado.columns and 'Epitope' in df_consolidado.columns:
    df_consolidado = df_consolidado.drop(columns=['Epitope sequence'])

df_consolidado

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils,Epitope name,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,alelos,percent_match
0,GSGIFITDNVHTWTEQY,16,32,CEEEEEECCCCCCCCCC,11,0,6,64.705882,NP 7,17,97.00% (840/866),94.12%,100.00%,2,0.480,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",97.00
1,GSGIFITDNVHTWTEQ,16,31,CEEEEEECCCCCCCCC,10,0,6,62.500000,NP 1,16,97.00% (840/866),93.75%,100.00%,2,0.330,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",97.00
2,GSGIFITDNVHTWTEQYK,16,33,CEEEEEECCCCCCCCCCE,11,0,7,61.111111,NP 11,18,96.77% (838/866),94.44%,100.00%,2,0.680,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",96.77
3,GSGIFITDNVHTWTE,16,30,CEEEEEECCCCCCCC,9,0,6,60.000000,NP 6,15,97.00% (840/866),93.33%,100.00%,2,0.445,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",97.00
4,GSGIFITDNVHTWT,16,29,CEEEEEECCCCCCC,8,0,6,57.142857,NP 12,14,97.00% (840/866),92.86%,100.00%,2,0.695,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",97.00


In [101]:
epitopos_filtrados_flex.to_csv("epitopos_validados_flexibilidade_tcell2.csv", index=False)